#1: Set Up the Environment

In [1]:
!pip install -q transformers datasets accelerate scikit-learn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
import torch
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports complete")

✅ Imports complete


# 2: Load and Explore Your Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')


file_path = '/content/drive/MyDrive/Toxic Video Game Chat Prediction Datasets/train.csv'

print("Loading first 50,000 rows...")
df = pd.read_csv(file_path, nrows=50000)
print(f"Loaded {len(df):,} rows")

print(f"\nColumns: {df.columns.tolist()}")

label_cols = ['severe_toxicity', 'obscene', 'identity_attack', 'insult', 'threat']

missing = [col for col in label_cols if col not in df.columns]
if missing:
    print(f"❌ Missing columns: {missing}")
    print("Available columns:", df.columns.tolist())
    raise ValueError("Update label_cols with correct names")

for col in label_cols:
    if df[col].dtype == float:
        df[col] = (df[col] > 0.5).astype(int)

print("\nLabel distribution (first 50k rows):")
for col in label_cols:
    positive = df[col].sum()
    pct = df[col].mean()
    print(f"  {col}: {positive:,} positive ({pct:.2%})")

Mounted at /content/drive
Loading first 50,000 rows...
Loaded 50,000 rows

Columns: ['id', 'target', 'comment_text', 'severe_toxicity', 'obscene', 'identity_attack', 'insult', 'threat', 'asian', 'atheist', 'bisexual', 'black', 'buddhist', 'christian', 'female', 'heterosexual', 'hindu', 'homosexual_gay_or_lesbian', 'intellectual_or_learning_disability', 'jewish', 'latino', 'male', 'muslim', 'other_disability', 'other_gender', 'other_race_or_ethnicity', 'other_religion', 'other_sexual_orientation', 'physical_disability', 'psychiatric_or_mental_illness', 'transgender', 'white', 'created_date', 'publication_id', 'parent_id', 'article_id', 'rating', 'funny', 'wow', 'sad', 'likes', 'disagree', 'sexual_explicit', 'identity_annotator_count', 'toxicity_annotator_count']

Label distribution (first 50k rows):
  severe_toxicity: 1 positive (0.00%)
  obscene: 280 positive (0.56%)
  identity_attack: 117 positive (0.23%)
  insult: 1,783 positive (3.57%)
  threat: 91 positive (0.18%)


# 3: Preprocess the Dataset

In [3]:
label_cols = ['severe_toxicity', 'obscene', 'identity_attack', 'insult', 'threat']

for col in label_cols:
    if df[col].dtype == float:
        df[col] = (df[col] > 0.5).astype(int)

print("Label distribution in loaded 50k rows:")
for col in label_cols:
    print(f"  {col}: {df[col].sum():,} positive ({df[col].mean():.2%})")

SAMPLE_SIZE = 50000
df_sample = df.sample(n=SAMPLE_SIZE, random_state=42).copy()
print(f"\nUsing {SAMPLE_SIZE:,} rows for training/evaluation")

df_sample['toxic_binary'] = (df_sample[label_cols].sum(axis=1) > 0).astype(int)
print(f"Toxic comments in sample: {df_sample['toxic_binary'].sum():,} "
      f"({df_sample['toxic_binary'].mean():.2%})")

X = df_sample['comment_text'].values
y = df_sample[label_cols].values.astype(np.float32)

stratify_col = df_sample['toxic_binary'].values
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify_col
)
print(f"Train before oversampling: {len(X_train):,}, Val: {len(X_val):,}")

train_toxic_mask = y_train.sum(axis=1) > 0
X_train_toxic = X_train[train_toxic_mask]
y_train_toxic = y_train[train_toxic_mask]
X_train_clean = X_train[~train_toxic_mask]
y_train_clean = y_train[~train_toxic_mask]

oversample_factor = 2
X_train = np.concatenate([X_train_clean] + [X_train_toxic] * (oversample_factor + 1))
y_train = np.concatenate([y_train_clean] + [y_train_toxic] * (oversample_factor + 1))

print(f"Train after oversampling: {len(X_train):,} (toxic now {y_train.sum(axis=1).sum():,} individual labels)")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = Dataset.from_dict({"text": X_train.tolist(), "labels": y_train.tolist()})
val_dataset   = Dataset.from_dict({"text": X_val.tolist(), "labels": y_val.tolist()})

train_dataset = train_dataset.map(tokenize_fn, batched=True)
val_dataset   = val_dataset.map(tokenize_fn, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"Label dtype: {train_dataset[0]['labels'].dtype}")
print("✅ Preprocessing & tokenisation complete")

Label distribution in loaded 50k rows:
  severe_toxicity: 1 positive (0.00%)
  obscene: 280 positive (0.56%)
  identity_attack: 117 positive (0.23%)
  insult: 1,783 positive (3.57%)
  threat: 91 positive (0.18%)

Using 50,000 rows for training/evaluation
Toxic comments in sample: 2,076 (4.15%)
Train before oversampling: 40,000, Val: 10,000
Train after oversampling: 43,322 (toxic now 5,454.0 individual labels)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/43322 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Label dtype: torch.float32
✅ Preprocessing & tokenisation complete


# 4: Tokenization (DistilBERT)

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_cols),
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

labels_tensor = torch.tensor(y_train)
pos_counts = labels_tensor.sum(dim=0)
neg_counts = len(labels_tensor) - pos_counts
pos_weight = neg_counts / pos_counts
pos_weight = pos_weight.to(device)
print(f"pos_weight per class: {pos_weight}")

from torch.nn import BCEWithLogitsLoss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = BCEWithLogitsLoss(pos_weight=pos_weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits))
    preds = (probs > 0.5).int().numpy()
    f1 = f1_score(labels, preds, average='micro')
    acc = accuracy_score(labels, preds)
    return {'f1_micro': f1, 'accuracy': acc}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

print("🚀 Starting training with weighted loss (3 epochs)...")
trainer.train()
print("✅ Training complete")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


pos_weight per class: tensor([     inf,  62.0597, 159.4519,   9.1196, 199.5648], device='cuda:0')
🚀 Starting training with weighted loss (3 epochs)...


Epoch,Training Loss,Validation Loss,F1 Micro,Accuracy
1,0.000000,nan,0.000000,0.958500
2,0.000000,nan,0.000000,0.958500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


✅ Training complete


# 5: Define Metrics and Train the Model

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_cols),
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits))
    preds = (probs > 0.5).int().numpy()
    f1 = f1_score(labels, preds, average='micro')
    acc = accuracy_score(labels, preds)
    return {'f1_micro': f1, 'accuracy': acc}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

print("🚀 Starting training (2 epochs, should take ~20-30 min on 20k samples)...")
trainer.train()
print("✅ Training complete!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Starting training (2 epochs, should take ~20-30 min on 20k samples)...


Epoch,Training Loss,Validation Loss,F1 Micro,Accuracy
1,0.024709,0.023810,0.597222,0.961000
2,0.006106,0.031158,0.591961,0.968100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


✅ Training complete!


# 6: Evaluate on Validation Set

In [8]:
preds_output = trainer.predict(val_dataset)
probs = torch.sigmoid(torch.tensor(preds_output.predictions))
preds = (probs > 0.5).int().numpy()

print("\nClassification Report:")
print(classification_report(val_dataset['labels'], preds, target_names=label_cols, zero_division=0))


Classification Report:
                 precision    recall  f1-score   support

severe_toxicity       0.00      0.00      0.00         1
        obscene       0.64      0.67      0.65        51
identity_attack       0.33      0.37      0.35        27
         insult       0.56      0.71      0.62       356
         threat       0.32      0.32      0.32        19

      micro avg       0.54      0.66      0.60       454
      macro avg       0.37      0.41      0.39       454
   weighted avg       0.54      0.66      0.59       454
    samples avg       0.03      0.03      0.03       454



# 7: Test with Your Own Comments


In [9]:
def predict_toxicity(comment):
    inputs = tokenizer(comment, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    results = {label: prob for label, prob in zip(label_cols, probs)}
    return results

test_comments = [
    "You are an amazing player! Great game!",
    "I hate you, you stupid noob!",
    "GG well played everyone",
    "You suck at this game, uninstall!",
    "Thanks for the help!",
    "This team is trash, everyone is terrible"
]

for comment in test_comments:
    print(f"\nComment: {comment}")
    scores = predict_toxicity(comment)
    for label, score in scores.items():
        print(f"  {label}: {score:.4f}")


Comment: You are an amazing player! Great game!
  severe_toxicity: 0.0000
  obscene: 0.0001
  identity_attack: 0.0002
  insult: 0.0089
  threat: 0.0007

Comment: I hate you, you stupid noob!
  severe_toxicity: 0.0013
  obscene: 0.0290
  identity_attack: 0.0187
  insult: 0.9944
  threat: 0.0062

Comment: GG well played everyone
  severe_toxicity: 0.0000
  obscene: 0.0002
  identity_attack: 0.0002
  insult: 0.0014
  threat: 0.0010

Comment: You suck at this game, uninstall!
  severe_toxicity: 0.0005
  obscene: 0.4606
  identity_attack: 0.0074
  insult: 0.9513
  threat: 0.0026

Comment: Thanks for the help!
  severe_toxicity: 0.0000
  obscene: 0.0001
  identity_attack: 0.0003
  insult: 0.0015
  threat: 0.0021

Comment: This team is trash, everyone is terrible
  severe_toxicity: 0.0003
  obscene: 0.0058
  identity_attack: 0.0072
  insult: 0.9922
  threat: 0.0020


# Evaluation Metrics

In [10]:

print("="*60)
print("EVALUATING MODEL ON VALIDATION SET")
print("="*60)

pred_output = trainer.predict(val_dataset)
logits = pred_output.predictions
true_labels = pred_output.label_ids

probs = torch.sigmoid(torch.tensor(logits))
pred_labels = (probs > 0.5).int().numpy()

true_labels_np = np.array(true_labels)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(true_labels_np, pred_labels)
precision_micro = precision_score(true_labels_np, pred_labels, average='micro', zero_division=0)
recall_micro = recall_score(true_labels_np, pred_labels, average='micro', zero_division=0)
f1_micro = f1_score(true_labels_np, pred_labels, average='micro', zero_division=0)

precision_macro = precision_score(true_labels_np, pred_labels, average='macro', zero_division=0)
recall_macro = recall_score(true_labels_np, pred_labels, average='macro', zero_division=0)
f1_macro = f1_score(true_labels_np, pred_labels, average='macro', zero_division=0)

print("\n📈 OVERALL METRICS")
print("-" * 40)
print(f"Accuracy:           {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision (micro):  {precision_micro:.4f}")
print(f"Recall (micro):     {recall_micro:.4f}")
print(f"F1-score (micro):   {f1_micro:.4f}")
print(f"\nPrecision (macro):  {precision_macro:.4f}")
print(f"Recall (macro):     {recall_macro:.4f}")
print(f"F1-score (macro):   {f1_macro:.4f}")

print("\n📋 PER-CLASS CLASSIFICATION REPORT")
print("-" * 40)
print(classification_report(
    true_labels_np,
    pred_labels,
    target_names=label_cols,
    zero_division=0
))

print("\n🔍 CONFUSION MATRICES (per toxicity type)")
print("-" * 40)
for i, label in enumerate(label_cols):
    tn, fp, fn, tp = confusion_matrix(true_labels_np[:, i], pred_labels[:, i]).ravel()

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    print(f"\n{label}:")
    print(f"  True Positives:  {tp:5d} | False Positives: {fp:5d}")
    print(f"  False Negatives: {fn:5d} | True Negatives:  {tn:5d}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-score:  {f1:.4f}")




print("\n✅ Evaluation complete!")

EVALUATING MODEL ON VALIDATION SET



📈 OVERALL METRICS
----------------------------------------
Accuracy:           0.9610 (96.10%)
Precision (micro):  0.5433
Recall (micro):     0.6630
F1-score (micro):   0.5972

Precision (macro):  0.3692
Recall (macro):     0.4116
F1-score (macro):   0.3884

📋 PER-CLASS CLASSIFICATION REPORT
----------------------------------------
                 precision    recall  f1-score   support

severe_toxicity       0.00      0.00      0.00         1
        obscene       0.64      0.67      0.65        51
identity_attack       0.33      0.37      0.35        27
         insult       0.56      0.71      0.62       356
         threat       0.32      0.32      0.32        19

      micro avg       0.54      0.66      0.60       454
      macro avg       0.37      0.41      0.39       454
   weighted avg       0.54      0.66      0.59       454
    samples avg       0.03      0.03      0.03       454


🔍 CONFUSION MATRICES (per toxicity type)
----------------------------------------

severe_t